# RSNA Knee | Which LLM Labels Can You Trust?

> **Answer.** For most findings, the choice among public RSNA knee LLM label sets barely matters.
> On gold-58, 11 of 12 findings show no clear difference between the best and the median source. MCL has 9 positives, too few to rank sources. On gold-74 all 12 findings show no clear difference.
> Even best against worst, only 1 of 12 findings separates on gold-58 (ACL: pilkwang above lixin).
> The CC0 consensus (plain mean of six sources; five on gold-58, since dread has no gold-58 rows) has a confidence interval that overlaps the best single source on 12 of 12 findings. Macro AUC on gold-58: consensus 0.8887, steven_v4 0.8927, steven_v2 0.8873.
> This compares labels with the 58 gold studies. It does not measure leaderboard impact.
>
> Dataset with the consensus file and all metrics: [RSNA Knee: Public LLM Label Sets Compared](https://www.kaggle.com/datasets/karttikjangid05/rsna-knee-public-llm-label-sets-compared)

## Setup
Every number below is computed in this notebook. Gold labels are used only for aggregate metrics. Nothing per study from the gold sets is printed or saved.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

ROOT = os.environ.get("AUDIT_ROOT", "/kaggle/input")
OUT = os.environ.get("AUDIT_OUT", "/kaggle/working")
UID = "StudyInstanceUID"
T = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
     "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
B, SEED = 2000, 42
print("bootstrap resamples:", B, "| seed:", SEED)


def find(name, required=True):
    hits = []
    for depth in range(1, 5):
        hits += glob.glob(os.path.join(ROOT, *["*"] * depth, name))
    hits = sorted(set(hits))
    if required:
        assert hits, f"{name} not found under {ROOT}"
    return hits[0] if hits else None


comp_dir = None
for c in ["rsna-knee-abnormality-detection", "competitions/rsna-knee-abnormality-detection"]:
    if os.path.exists(os.path.join(ROOT, c, "train.csv")):
        comp_dir = os.path.join(ROOT, c)
assert comp_dir, "competition data not attached"
print("competition data:", comp_dir)

## Sources and licences
Licences were read from each dataset's Kaggle metadata on 2026-10-09.
Only CC0 sources feed the consensus file. gperdrizet's set is CC BY-NC-SA 4.0, so its labels are not redistributed here. It is only scored.

In [ ]:
INV = [
    # id, owner, dataset, file, licence, llm, tuned_on_gold, note
    ("steven_v2", "stevenleehans", "stevenleehans/rsna-knee-llm-report-labels", "llm_labels_v2.csv", "CC0-1.0", "LLM (not named on card)", False, "own read; Synovitis fill from Effusion was motivated by gold-58 analysis"),
    ("steven_v4", "stevenleehans", "stevenleehans/rsna-knee-llm-report-labels", "llm_labels_v4_blend.csv", "CC0-1.0", "blend of steven_v2 and lixin", False, "blend of two included sources"),
    ("pilkwang", "pilkwang", "pilkwang/rsna-knee-llm-labels", "report_labels_v2.csv", "CC0-1.0", "LLM (not named on card)", False, "own read"),
    ("lixin", "lixin73", "lixin73/rsna-knee-llm-report-labels-sol56", "labels_llm_gpt56sol.csv", "CC0-1.0", "GPT-5.6-Sol", False, "own read"),
    ("qwen", "laymond (Raymond Yuen)", "laymond/rsna-knee-abnormality-qwen3-8b-weak-labels", "weak_labels_v2_raw.csv", "CC0-1.0", "Qwen2.5-32B-Instruct", False, "raw output, <finding>__confidence; slug says qwen3-8b, card says v2 files are Qwen2.5-32B"),
    ("yunus", "yunusgmsoy", "yunusgmsoy/rsna-knee-llm-report-labels", "yunus_llm_labels.csv", "CC0-1.0", "Gemini Flash", False, "own read"),
    ("dread", "dreaddevelopment", "dreaddevelopment/rsna-knee-labels", "labels_llm_soft.csv", "CC0-1.0", "LLM (not named on card)", False, "gold-58 rows not in file"),
    ("jev_final2", "riadmohamed42", "riadmohamed42/jev-knee-labels", "FINAL2_labels_scores.csv", "CC0-1.0", "OpenJev, 6-member ensemble", True, "member weights per finding from gold AUC"),
    ("jev_hybrid", "riadmohamed42", "riadmohamed42/jev-knee-labels", "HYBRID_labels_scores.csv", "CC0-1.0", "0.95 per-finding pick of steven_v4/pilkwang + 0.05 jev_final2", True, "column choice per finding on gold-58"),
    ("gperdrizet", "gperdrizet", "gperdrizet/rsna-knee-abnormality-pseudolabeled-training-data", "pseudolabeled-train.csv", "CC-BY-NC-SA-4.0", "Qwen2.5-7B + Phi4 vote", True, "per-label combination rule chosen on gold; file includes the gold-58 studies with their gold labels (label_source = gold), so those rows were dropped"),
]
VERDICT_POOL = ["steven_v2", "steven_v4", "pilkwang", "lixin", "qwen", "yunus"]
CONSENSUS = ["steven_v2", "pilkwang", "lixin", "qwen", "yunus", "dread"]
inv = pd.DataFrame(INV, columns=["source", "owner", "dataset", "file", "licence", "llm", "tuned_on_gold", "note"])
inv["link"] = "https://www.kaggle.com/datasets/" + inv["dataset"]
inv["in_verdict_pool"] = inv["source"].isin(VERDICT_POOL)
inv["in_consensus"] = inv["source"].isin(CONSENSUS)
assert (inv.loc[inv.in_consensus, "licence"] == "CC0-1.0").all()


def load(src, fname):
    path = find(fname)
    if src == "qwen":
        d = pd.read_csv(path, usecols=[UID] + [f"{t}__confidence" for t in T])
        d.columns = [UID] + T
    elif src == "gperdrizet":
        d = pd.read_csv(path, usecols=[UID, "label_source"] + T)
        d = d[d["label_source"] != "gold"].drop(columns="label_source")
    else:
        d = pd.read_csv(path, usecols=[UID] + T)
    d[UID] = d[UID].astype(str)
    assert d[UID].is_unique, src
    return d.set_index(UID)[T].astype(float)


S = {r.source: load(r.source, r.file) for r in inv.itertuples()}
print({k: v.shape for k, v in S.items()})

## Gold sets
gold-58 is the primary ruler: the 58 training studies with annotator labels in `train.csv`.
gold-74 adds the 16 studies JEV labelled by reading the reports. Report-based sources should look better on those 16, so gold-74 is secondary.

In [ ]:
train = pd.read_csv(os.path.join(comp_dir, "train.csv"), usecols=[UID] + T)
train[UID] = train[UID].astype(str)
lab = train.set_index(UID)[T]
g58 = lab[lab.notna().any(axis=1)].astype(float)
print("train studies:", len(lab), "| gold-58 studies:", len(g58), "| fully labelled:", int(g58.notna().all(axis=1).sum()))
assert len(g58) == 58 and set(np.unique(g58.values[~np.isnan(g58.values)])) <= {0.0, 1.0}

j74 = pd.read_csv(find("gold74_eval_set.csv"))
j74[UID] = j74[UID].astype(str)
extra = j74[j74["source"] != "original_gold58"].set_index(UID)[T].astype(float)
assert len(extra) == 16 and not set(extra.index) & set(g58.index)
g74 = pd.concat([g58, extra])
GOLD = {"gold58": g58, "gold74": g74}
for k, g in GOLD.items():
    print(k, "positives per finding:", g.sum().astype(int).to_dict())

## Coverage and alignment
All files use `StudyInstanceUID` and the competition's 12 finding names, so they merge directly with `train.csv`.

In [ ]:
all_ids = set(lab.index)
cov = []
for s, d in S.items():
    vals = d.values[~np.isnan(d.values)]
    cov.append({"source": s, "studies": len(d), "ids_not_in_train": len(set(d.index) - all_ids),
                "findings": int(d.notna().any().sum()), "empty_cells": int(d.isna().sum().sum()),
                "binary": bool(set(np.unique(vals)) <= {0.0, 1.0}),
                "gold58_rows": len(set(d.index) & set(g58.index)), "gold74_rows": len(set(d.index) & set(g74.index)),
                "exact_gold_match_share": float(np.nanmean((d.reindex(g58.index)[T].values == g58.values)[~np.isnan(g58.values)]))
                if len(set(d.index) & set(g58.index)) else np.nan})
cov = pd.DataFrame(cov).set_index("source")
print(cov.to_string())
assert (cov["ids_not_in_train"] == 0).all()
print("sources with over 95% of gold-58 cells equal to gold:", cov.index[cov["exact_gold_match_share"] > 0.95].tolist())

`exact_gold_match_share` checks that no scored file has the gold labels pasted in. For a binary source, AUC equals balanced accuracy at its single threshold.

## Consensus labels
One global rule, set before the first run that computed gold metrics: the plain mean of the CC0 sources' probabilities, with no weights.
dread has no gold-58 rows, so on gold-58 the consensus averages 5 sources (4 for the one study pilkwang lacks).
This notebook chooses no weight, threshold or source per finding on gold. JEV found that a per-finding blend picked on gold scored higher on the tuning set and lower on a holdout, so per-finding choices on 58 studies mostly fit noise.
The majority vote (share of sources at 0.5 or above) is reported as a second score. It is not in the published file.

In [ ]:
stack = {s: S[s].reindex(sorted(all_ids)) for s in CONSENSUS}
arr = np.stack([stack[s].values for s in CONSENSUS])
cons = pd.DataFrame(np.nanmean(arr, axis=0), index=sorted(all_ids), columns=T)
vote = pd.DataFrame(np.nanmean(np.where(np.isnan(arr), np.nan, arr >= 0.5), axis=0), index=cons.index, columns=T)
n_src = pd.Series(np.isfinite(arr).any(axis=2).sum(axis=0), index=cons.index, name="n_sources")
cons.index.name = UID
print("consensus rows:", len(cons), "| n_sources counts:", n_src.value_counts().sort_index().to_dict())
SCORED = {**S, "consensus_mean": cons, "consensus_vote": vote}

## Method
For each source, finding and gold set: ROC AUC with a 95% percentile confidence interval from a class-stratified bootstrap (2,000 resamples, seed 42).
A finding with fewer than 10 positives (or negatives) is flagged and not ranked.
Verdict per finding, among six sources that did not pick a source or weight per finding on gold-58: "clear winner" if the best source's lower CI bound is above the upper bound of the median source (the third best of six). Otherwise "no clear difference".
Overlapping intervals is a strict test. It favours "no clear difference".
As a descriptive extra (not part of the pre-set rule), the verdict table also says whether the best source is clearly above the worst one.

In [ ]:
def auc_ci(y, s, rng):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan, np.nan
    C = (pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :])
    ip = rng.integers(0, len(pos), (B, len(pos)))
    ineg = rng.integers(0, len(neg), (B, len(neg)))
    boot = C[ip[:, :, None], ineg[:, None, :]].mean(axis=(1, 2))
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return C.mean(), lo, hi


rows = []
for gi, (gname, g) in enumerate(GOLD.items()):
    for si, (s, d) in enumerate(SCORED.items()):
        dd = d.reindex(g.index)
        for fi, t in enumerate(T):
            ok = g[t].notna() & dd[t].notna()
            y, x = g.loc[ok, t].values, dd.loc[ok, t].values
            rng = np.random.default_rng([SEED, gi, fi])
            a, lo, hi = auc_ci(y, x, rng) if ok.sum() else (np.nan,) * 3
            rows.append({"source": s, "finding": t, "gold_set": gname, "n": int(ok.sum()),
                         "n_pos": int(y.sum()), "n_neg": int(len(y) - y.sum()), "auc": a, "ci_low": lo, "ci_high": hi})
M = pd.DataFrame(rows)


def flag(r):
    if r.n == 0:
        return "not scored: no gold rows"
    if r.n_pos < 10:
        return "too few positives to rank sources"
    if r.n_neg < 10:
        return "too few negatives to rank sources"
    return ""


M["flag"] = M.apply(flag, axis=1)
print(M[M.gold_set == "gold58"].groupby("finding")[["n_pos", "n_neg"]].max().reindex(T).to_string())

In [ ]:
# macro AUC (mean over findings) with a study-level bootstrap, for a one-line summary per source
mac = []
for gname, g in GOLD.items():
    for s, d in SCORED.items():
        dd = d.reindex(g.index)
        ok_rows = dd.notna().any(axis=1).values
        gg, xx = g.values[ok_rows], dd.values[ok_rows]
        if len(gg) == 0:
            continue

        def macro(idx):
            vals = []
            for j in range(len(T)):
                yy, ss = gg[idx, j], xx[idx, j]
                m = ~np.isnan(yy) & ~np.isnan(ss)
                if m.sum() and 0 < yy[m].sum() < m.sum():
                    vals.append(roc_auc_score(yy[m], ss[m]))
            return np.mean(vals) if len(vals) == len(T) else np.nan

        rng = np.random.default_rng(SEED)
        boot = np.array([macro(rng.integers(0, len(gg), len(gg))) for _ in range(B)])
        lo, hi = np.nanpercentile(boot, [2.5, 97.5])
        mac.append({"source": s, "finding": "macro", "gold_set": gname, "n": len(gg), "n_pos": np.nan, "n_neg": np.nan,
                    "auc": macro(np.arange(len(gg))), "ci_low": lo, "ci_high": hi, "flag": "study-level bootstrap" + (f"; only {len(gg)} studies, do not compare" if len(gg) < 30 else "")})
M = pd.concat([M, pd.DataFrame(mac)], ignore_index=True)
M = M.merge(inv[["source", "tuned_on_gold", "in_verdict_pool", "in_consensus"]], on="source", how="left")
M[["tuned_on_gold", "in_verdict_pool", "in_consensus"]] = M[["tuned_on_gold", "in_verdict_pool", "in_consensus"]].fillna(False).astype(bool)
print(M[M.finding == "macro"].pivot(index="source", columns="gold_set", values="auc").round(4).to_string())

## Verdict per finding

In [ ]:
ver = []
for gname in GOLD:
    for t in T:
        p = M[(M.gold_set == gname) & (M.finding == t) & M.source.isin(VERDICT_POOL) & (M.n > 0)].sort_values("auc", ascending=False)
        c = M[(M.gold_set == gname) & (M.finding == t) & (M.source == "consensus_mean")].iloc[0]
        best, med, worst = p.iloc[0], p.iloc[(len(p) - 1) // 2], p.iloc[-1]
        npos, nneg = int(GOLD[gname][t].sum()), int((GOLD[gname][t] == 0).sum())
        if npos < 10:
            v = "too few positives to rank sources"
        elif nneg < 10:
            v = "too few negatives to rank sources"
        elif best.ci_low > med.ci_high:
            v = "clear winner"
        else:
            v = "no clear difference"
        ver.append({"gold_set": gname, "finding": t, "n_pos": npos, "n_neg": nneg, "verdict": v,
                    "best_vs_worst_clear": bool(best.ci_low > worst.ci_high), "worst": worst.source,
                    "best": best.source, "best_auc": best.auc, "best_ci": f"{best.ci_low:.3f}-{best.ci_high:.3f}",
                    "median": med.source, "median_auc": med.auc, "median_ci": f"{med.ci_low:.3f}-{med.ci_high:.3f}",
                    "consensus_auc": c.auc, "consensus_ci": f"{c.ci_low:.3f}-{c.ci_high:.3f}",
                    "consensus_ci_overlaps_best": bool(c.ci_high >= best.ci_low and best.ci_high >= c.ci_low)})
V = pd.DataFrame(ver)
M = M.merge(V[["gold_set", "finding", "verdict"]], on=["gold_set", "finding"], how="left")
pd.set_option("display.width", 250)
for gname in GOLD:
    print(f"\n== {gname}")
    print(V[V.gold_set == gname].drop(columns="gold_set").round(3).to_string(index=False))
    print("verdict counts:", V[V.gold_set == gname].verdict.value_counts().to_dict())
    print("best source clearly above worst source (descriptive, not pre-registered):", int(V[V.gold_set == gname].best_vs_worst_clear.sum()), "of", len(T))
    print("consensus CI overlaps best single source:", int(V[V.gold_set == gname].consensus_ci_overlaps_best.sum()), "of", len(T))

## Main plot: AUC with 95% CI on gold-58

In [ ]:
order = VERDICT_POOL + ["jev_final2", "jev_hybrid", "consensus_vote", "consensus_mean"]
G58 = M[(M.gold_set == "gold58") & (M.finding != "macro")]
fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharey=True)
plt.rcParams["font.size"] = 10
for ax, t in zip(axes.ravel(), T):
    sub = G58[G58.finding == t].set_index("source").reindex(order)
    for i, s in enumerate(order):
        r = sub.loc[s]
        col = "#2a78d6" if s.startswith("consensus") else "#52514e"
        ax.plot([r.ci_low, r.ci_high], [i, i], color=col, lw=2, solid_capstyle="round")
        ax.plot(r.auc, i, "o", ms=8, color=col, mfc=col if not r.tuned_on_gold else "white", mew=2)
    v = V[(V.gold_set == "gold58") & (V.finding == t)].iloc[0]
    ax.set_title(f"{t}  (pos {v.n_pos})\n{v.verdict}", fontsize=10, color="#0b0b0b")
    ax.set_yticks(range(len(order)), order)
    ax.grid(axis="x", color="#e6e5e0", lw=0.8)
    ax.set_xlim(0.4, 1.01)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
fig.suptitle("Public RSNA knee label sets vs gold-58: ROC AUC with 95% bootstrap CI (hollow = tuned on gold, blue = consensus)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "label_auc_gold58.png"), dpi=110, facecolor="white")
plt.show()

## Results table (gold-58, AUC and 95% CI)

In [ ]:
tab = G58.assign(cell=G58.apply(lambda r: f"{r.auc:.3f} ({r.ci_low:.2f}-{r.ci_high:.2f})" if r.n else "", axis=1))
print(tab.pivot(index="finding", columns="source", values="cell").reindex(T)[[s for s in order if s in tab.source.unique()]].to_string())

## Output files

In [ ]:
pub = cons.round(4).join(n_src)
pub.to_csv(os.path.join(OUT, "consensus_labels.csv"))
M.to_csv(os.path.join(OUT, "source_metrics.csv"), index=False)
inv_out = inv.merge(cov[["studies", "gold58_rows", "gold74_rows", "binary"]], left_on="source", right_index=True)
inv_out.to_csv(os.path.join(OUT, "source_inventory.csv"), index=False)
print("wrote", pub.shape, M.shape, inv_out.shape)

## Data card tables
These are built from the saved `source_metrics.csv`, which holds unrounded values. The tables round them to 3 or 4 decimals, and the data card shows exactly these numbers.

In [ ]:
def card_tables(m, inv):
    T = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
    pool = inv.loc[inv.in_verdict_pool, "source"].tolist()
    g = m[m.gold_set == "gold58"]
    rows = ["| Finding | Gold-58 positives | Verdict | Best source (AUC, 95% CI) | Consensus (AUC, 95% CI) |", "|---|---|---|---|---|"]
    for t in T:
        f = g[g.finding == t]
        p = f[f.source.isin(pool)].sort_values("auc", ascending=False).iloc[0]
        c = f[f.source == "consensus_mean"].iloc[0]
        rows.append(f"| {t} | {int(f.n_pos.max())} | {f.verdict.iloc[0]} | {p.source} {p.auc:.3f} ({p.ci_low:.3f} to {p.ci_high:.3f}) | {c.auc:.3f} ({c.ci_low:.3f} to {c.ci_high:.3f}) |")
    tuned = set(inv.loc[inv.tuned_on_gold, "source"])
    mac = m[m.finding == "macro"].pivot(index="source", columns="gold_set", values="auc").sort_values("gold58", ascending=False)
    mrows = ["| Source | Macro AUC gold-58 | Macro AUC gold-74 |", "|---|---|---|"]
    for s, r in mac.iterrows():
        a = "not scored" if r.gold58 != r.gold58 else f"{r.gold58:.4f}"
        b = f"{r.gold74:.4f}" + (" (16 studies only)" if r.gold58 != r.gold58 else "")
        mrows.append(f"| {s}{' (tuned on gold)' if s in tuned else ''} | {a} | {b} |")
    irows = ["| Source | Owner | Licence | Studies | In consensus |", "|---|---|---|---|---|"]
    for r in inv.itertuples():
        irows.append(f"| [{r.source}]({r.link}) | {r.owner} | {r.licence} | {r.studies} | {'yes' if r.in_consensus else 'no'} |")
    return "\n".join(rows), "\n".join(mrows), "\n".join(irows)
m_saved = pd.read_csv(os.path.join(OUT, "source_metrics.csv"))
inv_saved = pd.read_csv(os.path.join(OUT, "source_inventory.csv"))
for block in card_tables(m_saved, inv_saved)[:2]:
    print(block, end="\n\n")

In [ ]:
# Reproduction check against the published dataset, when it is attached
pm = find("source_metrics.csv", required=False) if ROOT != OUT else None
if pm and os.path.dirname(pm) != OUT:
    old = pd.read_csv(pm)
    new = pd.read_csv(os.path.join(OUT, "source_metrics.csv"))
    oc = pd.read_csv(os.path.join(os.path.dirname(pm), "consensus_labels.csv"), dtype={UID: str}).set_index(UID)
    assert np.allclose(old[["auc", "ci_low", "ci_high"]].values, new[["auc", "ci_low", "ci_high"]].values, equal_nan=True)
    assert (old["verdict"].fillna("") == new["verdict"].fillna("")).all()
    assert np.allclose(oc[T].values, pub.loc[oc.index, T].values) and (oc["n_sources"] == pub.loc[oc.index, "n_sources"]).all()
    print("REPRODUCED: published source_metrics.csv and consensus_labels.csv match this run")
else:
    print("published dataset not attached; skipping reproduction check")

## How to use the consensus labels
```python
train = pd.read_csv(".../rsna-knee-abnormality-detection/train.csv")
cons = pd.read_csv(".../consensus_labels.csv")
df = train.merge(cons, on="StudyInstanceUID", suffixes=("", "_cons"))
```
Each finding column is a soft target in [0, 1]. `n_sources` says how many sources were averaged for that study. Keep the 58 gold studies out of training if you want to measure on them.

## Limits
- 58 studies is a small ruler. Many intervals in the table are wide, so differences of a few AUC points between sources are not resolved.
- Gold-74 adds 16 studies that JEV labelled by reading the reports. A report-based source has an edge on those, so gold-74 is only a secondary check.
- dread has no gold-58 rows, so it is scored only on the 16 extra studies. gperdrizet's file includes the gold-58 studies with their gold labels (marked label_source = gold), so those rows were dropped and it is scored only on the 16 extra studies. gperdrizet also chose its per-label rule on gold. 16 studies is too few to compare.
- jev_final2 and jev_hybrid made choices using gold studies, so their gold scores are optimistic. They are plotted with hollow markers and left out of the verdict.
- steven_v2 fills undecided Synovitis cells from Effusion. stevenleehans motivates this rule with gold-58 numbers, so it is a choice for one finding informed by gold. steven_v2's Synovitis score is likely optimistic, and steven_v4, jev_hybrid and the consensus inherit part of it. Treat the Synovitis results with care.
- Several other sources report their own gold-58 scores, so their authors may have used gold-58 for global choices. Those gold scores may be a little optimistic.
- On gold-58 the consensus averages 5 sources (4 for one study), because dread has no gold-58 rows.
- The consensus is not weighted. A better weighting may exist, but finding it on 58 studies would repeat the per-finding overfit this notebook avoids.
- This measures agreement with the gold labels. It does not measure leaderboard impact, and no leaderboard gain is claimed.

## Credit
- stevenleehans: [RSNA Knee LLM Report Labels](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels)
- Pilkwang Kim: [RSNA Knee LLM-read report labels](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels)
- lixin73: [RSNA Knee LLM Report Labels (GPT-5.6-Sol)](https://www.kaggle.com/datasets/lixin73/rsna-knee-llm-report-labels-sol56)
- Raymond Yuen: [Qwen2.5-32B Weak Labels](https://www.kaggle.com/datasets/laymond/rsna-knee-abnormality-qwen3-8b-weak-labels)
- yunusgmsoy: [RSNA Knee LLM Report Labels (Gemini Flash)](https://www.kaggle.com/datasets/yunusgmsoy/rsna-knee-llm-report-labels)
- dreaddevelopment: [Knee MRI LLM Report-Distilled Soft Labels](https://www.kaggle.com/datasets/dreaddevelopment/rsna-knee-labels)
- gperdrizet: [RSNA Knee Abnormality: pseudolabeled training data](https://www.kaggle.com/datasets/gperdrizet/rsna-knee-abnormality-pseudolabeled-training-data) (CC BY-NC-SA 4.0, scored only)
- riadmohamed42 (JEV): [JEV-Knee labels](https://www.kaggle.com/datasets/riadmohamed42/jev-knee-labels) and [verification notebook](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification). JEV built the 74-study evaluation set and showed that per-finding blending on gold overfits.
- Earlier comparisons: arin, [RSNA Knee: 58 Gold Labels vs 4 Teachers](https://www.kaggle.com/code/skarin/rsna-knee-58-gold-labels-vs-4-teachers), and starkhushi, [Which Report Labels Should You Train On](https://www.kaggle.com/code/starkhushi/rsna-knee-which-report-labels-should-you-train-on).

What this adds to JEV's work: JEV measured two public sets plus JEV's own ensemble on gold-74 and picked a winner per finding. This notebook scores eight sources on gold-58 with a confidence interval for every source and finding, keeps gold-74 as a secondary check, uses one global consensus rule with no per-finding tuning, and publishes the consensus as a CC0 file.